In [1]:
from dask.distributed import Client

client = Client("tcp://127.0.0.1:34279")
client

<Client: 'tcp://127.0.0.1:34279' processes=9 threads=72, memory=136.41 GiB>

In [2]:
import xarray as xr
import numpy as np
import os.path as op
import tripyview as tpv
from xgcm.grid import Grid
import gsw
import matplotlib.pyplot as plt

# xr.set_options(keep_attrs=True)
# do_parallel       = True
# parallel_nprc     = 16   # number of dask workers
# parallel_nprc_bin = 8   # number of processor used to parallize the binning loop
# parallel_tmem     = 256   # max. available RAM

%matplotlib inline

/work/ba1254/b383315/conda/envs/fesom/lib/python3.12/site-packages/tripyview/sub_data.py:8: UserWarning: The seawater library is deprecated! Please use gsw instead.
  import seawater as sw


/work/ba1254/b383315/conda/envs/fesom/lib/python3.12/site-packages


In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
ddir = "/work/uo0119/a270067/runtime/awicm3-v3.1/"
sdir = "/scratch/b/b383315/fesom2/next/"
wdir = "/work/ba1254/b383315/fesom2/next/"

In [5]:
plt_contb = False        # background contour line (thin)
plt_contf = False       # contour line of main colorbar steps 
plt_contr = False       # contour line of reference value 
plt_contl = False       # label contourline of main colorbar steps 
do_plt = 'tpc'          # plot pcolor (tpc) or contourf (tcf)
box = [-180, 180, -90, 90] # regional limitation of plot. [lonmin, lonmax, latmin, latmax]
proj = 'rob'               # Robinson projection
# proj = 'sps'               # SouthPolarStereo projection
do_rescale = None          # rescale data: None, 'log10', 'slog10', np.array(...)
nrow0 = 1                  # int, (default: 1) number of columns when plotting multiple data panels
ncol0 = 1                  # int, (default: 1) number of rows when plotting multiple data panels
do_enum = False            # bool, (default: False) do enumeration of axes with a), b), c) …
cbl_opt = dict()           # dict, (default: dict()) direct option for colorbar labels (fontsize, fontweight, …) via kwarg
cbtl_opt = dict()          # dict, (default: dict()) direct option for colorbar tick labels (fontsize, fontweight, …) via kwarg
save_dpi = 500

# cstr      = 'matplotlib.RdBu_r'
# cnum      = 30
# cref      = 0
crange    = None
cfac      = None
climit    = None
chist     = True
ctresh    = 0.995

# cb_label = "DS05 - clim salt annual"
cstr      = 'matplotlib.RdBu_r'
# cstr      = 'cmocean.balance'
cnum      = 41
cref      = 0.
cmin      = -1e-6
cmax      = 1e-6

cinfo = tpv.set_cinfo(cstr, cnum, crange, cmin, cmax, 
                      cref, cfac, climit, chist, ctresh
                     ) 

In [6]:
NXX = 50
ds1 = xr.open_dataset(op.join(ddir,"N%2d/outdata/fesom/unod.fesom.2090.nc"
                              % NXX),
                     chunks={"nz1":1,"x":100000}
                    )
ds = xr.open_dataset(op.join(ddir,"N%2d/outdata/fesom/w.fesom.2090.nc"
                             % NXX),
                     chunks={"nz":1,"x":100000}
                    )
dssalt = xr.open_dataset(op.join(ddir,"N%2d/outdata/fesom/salt.fesom.2090.nc"
                              % NXX),
                     chunks={"nz1":1,"x":100000}
                    ).rename_dims({'x':'nod2'})
ds1, ds

(<xarray.Dataset> Size: 29GB
 Dimensions:  (time: 12, nz1: 47, x: 12952215)
 Coordinates:
   * time     (time) datetime64[ns] 96B 2090-01-31T23:57:00 ... 2090-12-31T23:...
   * nz1      (nz1) float64 376B 2.5 7.5 15.0 ... 5.525e+03 5.825e+03 6.125e+03
 Dimensions without coordinates: x
 Data variables:
     unod     (time, nz1, x) float32 29GB dask.array<chunksize=(1, 1, 100000), meta=np.ndarray>
 Attributes: (12/21)
     CDI:                                 Climate Data Interface version 2.2.4...
     Conventions:                         CF-1.6
     FESOM_model:                         FESOM2
     FESOM_website:                       fesom.de
     FESOM_git_SHA:                       e23bc823
     FESOM_MeshPath:                      /work/ab0995/a270067/fesom2/next/mesh/
     ...                                  ...
     FESOM_evp_rheol_steps:               120
     FESOM_opt_visc:                      7
     FESOM_use_wsplit:                    -1
     FESOM_autorotate_back_to_geo: 

In [7]:
iz2000 = 31
iz4300 = 40
dz = xr.DataArray(np.diff(ds.nz), dims=["nz1"],
                  coords={"nz1":ds1.nz1}
                 ).isel(nz1=slice(None,iz4300))
dz

<xarray.DataArray (nz1: 40)> Size: 320B
array([  5.,   5.,  10.,  10.,  10.,  10.,  10.,  10.,  10.,  10.,  10.,
        15.,  20.,  25.,  30.,  40.,  50.,  60.,  70.,  80.,  90., 100.,
       110., 120., 130., 140., 150., 170., 200., 220., 230., 250., 250.,
       250., 250., 250., 250., 250., 250., 250.])
Coordinates:
  * nz1      (nz1) float64 320B 2.5 7.5 15.0 ... 3.775e+03 4.025e+03 4.275e+03

In [8]:
dt = 180. # 3 minutes

mesh_path = '/work/ab0995/a270067/fesom2/next/mesh/'
mesh = tpv.load_mesh_fesom2(mesh_path, do_rot='None', focus=0, 
                            do_info=True, do_pickle=False
                           )
mesh

 > load mesh from *.out files: /work/ab0995/a270067/fesom2/next/mesh
 > load e_area from fesom.mesh.diag.nc
 > load n_area from fesom.mesh.diag.nc
 > compute lsmask
 > save *.shp to /home/b/b383315/meshcache_tripyview/mesh/tripyview_fesom2_mesh_pbnd.shp
 > augment lsmask
 > save *.shp to /home/b/b383315/meshcache_tripyview/mesh/tripyview_fesom2_mesh_focus=0.shp
___FESOM2 MESH INFO________________________
 > path            = /work/ab0995/a270067/fesom2/next/mesh
 > id              = mesh
 > do rot          = None
 > [al,be,ga]      = 50, 15, -90
 > do augmpbnd     = True
 > do cavity       = False
 > do lsmask       = True
 > do earea,eresol = True, False
 > do narea,nresol = True, False
___________________________________________
 > #node           = 12952215
 > #elem           = 25742475
 > #lvls           = 48
___________________________________________


___FESOM2 MESH INFO________________________
 > path            = /work/ab0995/a270067/fesom2/next/mesh
 > id              = mesh
 > do rot          = None
 > [al,be,ga]      = 50, 15, -90
 > do augmpbnd     = True
 > do cavity       = False
 > do lsmask       = True
 > do earea,eresol = True, False
 > do narea,nresol = True, False
___________________________________________
 > #node           = 12952215
 > #elem           = 25742475
 > #lvls           = 48
___________________________________________

In [9]:
diag = xr.open_dataset(op.join(mesh_path, 'fesom.mesh.diag.nc'),
                       chunks={'nz':1,"nz1":1,'elem':200000,
                               'nod2':100000,'edg_n':300000}
                      )

ddx = diag['gradient_sca_x']
ddy = diag['gradient_sca_y']
area = xr.DataArray(.5*(diag.nod_area.isel(nz=slice(None,-1)).data
                        + diag.nod_area.isel(nz=slice(1,None)).data
                       ),
                    dims=dssalt.salt.isel(time=0).dims, 
                    coords=dssalt.salt.isel(time=0).coords
                   ).chunk(dssalt.salt.isel(time=0).chunks
                          ).isel(nz1=slice(None,iz4300))
depth = diag.zbar_n_bottom
depth.coords["nod2"] = ("nod2",np.arange(len(diag.nod2)))
area.coords["nod2"] = ("nod2",np.arange(len(diag.nod2)))

diag

<xarray.Dataset> Size: 10GB
Dimensions:            (nz: 48, nz1: 47, elem: 25742475, nod2: 12952215, n3: 3,
                        n2: 2, N: 8, edg_n: 38695599, n4: 4)
Coordinates:
  * nz                 (nz) float64 384B 0.0 -5.0 -10.0 ... -6e+03 -6.25e+03
  * nz1                (nz1) float64 376B -2.5 -7.5 ... -5.825e+03 -6.125e+03
Dimensions without coordinates: elem, nod2, n3, n2, N, edg_n, n4
Data variables: (12/19)
    elem_area          (elem) float64 206MB dask.array<chunksize=(200000,), meta=np.ndarray>
    nlevels_nod2D      (nod2) int32 52MB dask.array<chunksize=(100000,), meta=np.ndarray>
    nlevels            (elem) int32 103MB dask.array<chunksize=(200000,), meta=np.ndarray>
    nod_in_elem2D_num  (nod2) int32 52MB dask.array<chunksize=(100000,), meta=np.ndarray>
    nod_part           (nod2) int32 52MB dask.array<chunksize=(100000,), meta=np.ndarray>
    elem_part          (elem) int32 103MB dask.array<chunksize=(200000,), meta=np.ndarray>
    ...                 ...
    nod_in_elem2D      (N, nod2) int32 414MB dask.array<chunksize=(8, 100000), meta=np.ndarray>
    edges              (n2, edg_n) int32 310MB dask.array<chunksize=(2, 300000), meta=np.ndarray>
    edge_tri           (n2, edg_n) int32 310MB dask.array<chunksize=(2, 300000), meta=np.ndarray>
    edge_cross_dxdy    (n4, edg_n) float64 1GB dask.array<chunksize=(4, 300000), meta=np.ndarray>
    gradient_sca_x     (elem, n3) float64 618MB dask.array<chunksize=(200000, 3), meta=np.ndarray>
    gradient_sca_y     (elem, n3) float64 618MB dask.array<chunksize=(200000, 3), meta=np.ndarray>

# Advection of $u$
## Saving 3D to scratch

In [ ]:
ystart = 2091
# years = np.array([2002,2005,2006,2007,2008])
NXX = 50

JFMJAS = np.concatenate((np.arange(1,4, dtype=int),
                         np.arange(7,10, dtype=int)))

for yy in range(ystart,2101):
# for yy in years:
    
    u_path = op.join(ddir,'N%02d/outdata/fesom/unod.fesom.%4d.nc'
                      % (NXX,yy))
    v_path = op.join(ddir,'N%02d/outdata/fesom/vnod.fesom.%4d.nc'
                      % (NXX,yy))
    w_path = op.join(ddir,'N%02d/outdata/fesom/w.fesom.%4d.nc'
                      % (NXX,yy))
    Av_path = op.join(ddir,'N%02d/outdata/fesom/ke_adv_v.fesom.%4d.nc'
                      % (NXX,yy))
        
    if yy == ystart:
        months = JFMJAS[:]
        print(months)
    else:
        months = JFMJAS
    
    for mm in months:
        
        try:
            divVbvb = xr.open_zarr(op.join(sdir,"ke_adv/divVbvb_%4d-%02d.zarr" 
                                           % (yy,mm))
                                  )
            print("Intermediary step exists :)")
            for iz in range(iz4300):
                if iz == 0:
                    vnod = tpv.load_data_fesom2(mesh, v_path, do_filename=True,
                                              vname="vnod", do_info=False,
                                              depth=[ds1.nz1[iz].values], year=yy, mon=[mm]
                                             ).chunk({'nod2':100000})
                else:
                    vnod = xr.concat([vnod, 
                                      tpv.load_data_fesom2(mesh, v_path, do_filename=True,
                                              vname="vnod", do_info=False,
                                              depth=[ds1.nz1[iz].values], year=yy, mon=[mm]
                                             ).chunk({'nod2':100000})
                                     ], "nz1")
        
        except:
            for iz in range(iz4300):
                dsu = xr.open_dataset(u_path
                                     ).isel(time=mm-1,nz1=iz)

                dsv = xr.open_dataset(v_path
                                     ).isel(time=mm-1,nz1=iz)

                vv = xr.DataArray((dsv["vnod"]**2).values[mesh.e_i],
                                  dims=["elem","n3"]
                                 ).chunk({"elem":200000})
                uv = xr.DataArray((dsu["unod"]*dsv["vnod"]).values[mesh.e_i],
                                  dims=["elem","n3"]
                                 ).chunk({"elem":200000})
                dsuv_e = (vv.data * ddy).sum("n3").to_dataset(name="uv_x")
                dsvv_e = (uv.data * ddx).sum("n3").to_dataset(name="vv_y")

                if iz == 0:
                    dsvv = tpv.sub_data.do_interp_e2n(dsvv_e, mesh, True
                                                     ).chunk({'nod2':100000})
                    dsuv = tpv.sub_data.do_interp_e2n(dsuv_e, mesh, True
                                                     ).chunk({'nod2':100000})
                    vnod = tpv.load_data_fesom2(mesh, v_path, do_filename=True,
                                              vname="vnod", do_info=False,
                                              depth=[ds1.nz1[iz].values], year=yy, mon=[mm]
                                             ).chunk({'nod2':100000})
                    w = tpv.load_data_fesom2(mesh, w_path, do_filename=True,
                                             vname="w", do_info=False,
                                             depth=[ds.nz[iz].values], year=yy, mon=[mm]
                                            ).chunk({'nod2':100000})
                else:
                    dsvv = xr.concat([dsvv,
                                      tpv.sub_data.do_interp_e2n(dsvv_e, mesh, True
                                                                ).chunk({'nod2':100000})
                                     ], "nz1")
                    dsuv = xr.concat([dsuv, 
                                      tpv.sub_data.do_interp_e2n(dsuv_e, mesh, True
                                                                ).chunk({'nod2':100000})
                                     ], "nz1")
                    vnod = xr.concat([vnod, 
                                      tpv.load_data_fesom2(mesh, v_path, do_filename=True,
                                              vname="vnod", do_info=False,
                                              depth=[ds1.nz1[iz].values], year=yy, mon=[mm]
                                             ).chunk({'nod2':100000})
                                     ], "nz1")
                    w = xr.concat([w, 
                                   tpv.load_data_fesom2(mesh, w_path, do_filename=True,
                                              vname="w", do_info=False,
                                              depth=[ds.nz[iz].values], year=yy, mon=[mm]
                                             ).chunk({'nod2':100000})
                                  ], "nz")

                dsvv_e.close(), dsuv_e.close()
                dsu.close(), dsv.close()

            vnod = xr.concat([vnod,
                              tpv.load_data_fesom2(mesh, v_path, do_filename=True,
                                              vname="vnod", do_info=False,
                                              depth=[ds1.nz1[iz4300].values], year=yy, mon=[mm]
                                             ).chunk({'nod2':100000})
                             ], "nz1")
            w = xr.concat([w, 
                           tpv.load_data_fesom2(mesh, w_path, do_filename=True,
                                              vname="w", do_info=False,
                                              depth=[ds.nz[iz4300].values], year=yy, mon=[mm]
                                             ).chunk({'nod2':100000})
                          ], "nz")

            dsvv.coords["nz1"] = ("nz1",dz.nz1.data)
            dsuv.coords["nz1"] = ("nz1",dz.nz1.data)

            V = xr.DataArray(vnod.vnod.data, 
                             dims=["nz1","nod2"],
                             coords={"nz1":ds1.nz1.isel(nz1=slice(None,iz4300+1)
                                                       ).values}
                            ).to_dataset(name="vnod")
            V["w"] = xr.DataArray(w.w.data,
                                  dims=["nz","nod2"],
                                  coords={"nz":ds.nz.isel(nz=slice(None,iz4300+1)
                                                         ).values}
                                 )

            grid = Grid(V, periodic=False,
                        coords={'Z': {'center': 'nz1', 'left': 'nz'}},
                        autoparse_metadata=False
                       )

            wbvb = V.w * grid.interp(V.vnod, "Z",
                                     boundary="extend"
                                    )
            wbvb_z = (-grid.diff(wbvb, "Z", 
                                 boundary="extend"
                                ).isel(nz1=slice(None,-1))
                      / dz)

            dsave = (dsuv.n_uv_x 
                     + dsvv.n_vv_y 
                     + wbvb_z
                    ).reset_coords(drop=True).chunk({"nz1":1,"nod2":100000}
                                                   ).to_dataset(name="divVv")
            dsave.to_zarr(op.join(sdir,"ke_adv/divVbvb_%4d-%02d.zarr" 
                                  % (yy,mm)),
                          mode='w')
            dsave.close(), V.close(), w.close()
            del wbvb, wbvb_z
            dsuv.close(), dsvv.close()
            print("iiiiiiiiiiiiiiiiiiiii")
            print("iiIntermediate step!!")
            print("!!!!!!!!!!!!!!!!!!!!!")

            divVbvb = xr.open_zarr(op.join(sdir,"ke_adv/divVbvb_%4d-%02d.zarr" 
                                           % (yy,mm))
                                  )
        
        for iz in range(iz4300):
            if iz == 0:
                Avnod = tpv.load_data_fesom2(mesh, Av_path, do_filename=True,
                                      vname="ke_adv_v", do_info=False,
                                      depth=[ds1.nz1[iz].values], year=yy, mon=[mm]
                                     ).chunk({'nod2':100000})
            else:
                Avnod = xr.concat([Avnod, 
                                   tpv.load_data_fesom2(mesh, Av_path, do_filename=True,
                                      vname="ke_adv_v", do_info=False,
                                      depth=[ds1.nz1[iz].values], year=yy, mon=[mm]
                                     ).chunk({'nod2':100000})
                                  ], "nz1")

        tv = (vnod.vnod
              * (-Avnod.n_ke_adv_v/dt - divVbvb.divVv)
             )
        
        dsave = tv.to_dataset(name="Tv")
        dsave.to_zarr(op.join(sdir,"ke_adv/N%2d/Tv_%4d-%02d.zarr" 
                              % (NXX,yy,mm)),
                      mode='w')
        dsave.close(), divVbvb.close()
        vnod.close(), Avnod.close()
        print("iiiiiiiiiiiiiiiiiiii")
        print("iiSaved to scratch!!")
        print("!!!!!!!!!!!!!!!!!!!!")
        del tv
        print("Month: ", mm)
    print("Year: ", yy)

[1 2 3 7 8 9]
 > do interpolation e2n
 > do interpolation e2n


/work/ba1254/b383315/conda/envs/fesom/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 294.68 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/work/ba1254/b383315/conda/envs/fesom/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/work/ba1254/b383315/conda/envs/fesom/lib/python3.12/site-packages/tripyview/sub_data.py:1002: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  if   (mon is not None): sel_mon = np.in1d( data['time.mon

 > do interpolation e2n


/work/ba1254/b383315/conda/envs/fesom/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 294.68 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/work/ba1254/b383315/conda/envs/fesom/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


 > do interpolation e2n


...

/work/ba1254/b383315/conda/envs/fesom/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 294.68 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/work/ba1254/b383315/conda/envs/fesom/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/work/ba1254/b383315/conda/envs/fesom/lib/python3.12/site-packages/tripyview/sub_data.py:1002: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  if   (mon is not None): sel_mon = np.in1d( data['time.mon

 > do interpolation e2n


/work/ba1254/b383315/conda/envs/fesom/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 294.68 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/work/ba1254/b383315/conda/envs/fesom/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


 > do interpolation e2n


/work/ba1254/b383315/conda/envs/fesom/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 294.68 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/work/ba1254/b383315/conda/envs/fesom/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Saving 2D to work

In [ ]:
ystart = 2003
NXX = 48

JFMJAS = np.concatenate((np.arange(1,4, dtype=int),
                         np.arange(7,10, dtype=int)))


# for yy in np.arange(ystart,2011,1, dtype=int):
for yy in range(ystart,2011):
    
    h_path = op.join(ddir,'N%02d/outdata/fesom/MLD3.fesom.%4d.nc'
                      % (NXX,yy))
        
    # if yy == ystart:
    #     months = JFMJAS[-2:]
    #     print(months)
    # else:
    months = JFMJAS
    
    for mm in months:
        
        tv = xr.open_zarr(op.join(sdir,"ke_adv/N%2d/Tv_%4d-%02d.zarr" 
                                       % (NXX,yy,mm))
                              )
        
        mld = tpv.load_data_fesom2(mesh, h_path, do_filename=True,
                                  vname="MLD3", do_info=False,
                                  year=yy, mon=[mm]
                                 ).chunk({'nod2':100000})
        
        dsave = ((tv.Tv
                  * dz
                 ).where(tu.nz1<np.abs(mld.MLD3)).sum("nz1",skipna=True) 
                 / np.abs(mld.MLD3)
                ).reset_coords(drop=True).to_dataset(name="Tv")
        dsave.to_zarr(op.join(wdir,"ke_adv/N%2d/MLD3/Tv_%4d-%02d.zarr" 
                              % (NXX,yy,mm)),
                      mode='w')
        dsave.close(), mld.close(), tv.close()
        print("iiiiiiiiiiiiiiiii")
        print("iiSaved to work!!")
        print("!!!!!!!!!!!!!!!!!")
        print("Month: ", mm)

    print("Year: ", yy)

In [ ]:
ystart = 2000
NXX = 48

JFMJAS = np.concatenate((np.arange(1,4, dtype=int),
                         np.arange(7,10, dtype=int)))

we, ea, so, no = (-80,-60,31,45) # GulfStrEx
# we, ea, so, no = (130,150,31,45)   # KuroEx
# we, ea, so, no = (-75,-45,-65,-55) # DrakePass
# we, ea, so, no = (-45,-35,10,25)   # Agulhas

# for yy in np.arange(ystart,2011,1, dtype=int):
for yy in range(ystart,2011):
    
    h_path = op.join(ddir,'N%02d/outdata/fesom/MLD2.fesom.%4d.nc'
                      % (NXX,yy))
        
    # if yy == ystart:
    #     months = JFMJAS[-2:]
    #     print(months)
    # else:
    months = JFMJAS
    
    for mm in months:
        
        tu = xr.open_zarr(op.join(sdir,"ke_adv/N%2d/Tu_%4d-%02d.zarr" 
                                       % (NXX,yy,mm))
                              )
        
        mld = tpv.load_data_fesom2(mesh, h_path, do_filename=True,
                                  vname="MLD2", do_info=False,
                                  year=yy, mon=[mm]
                                 ).chunk({'nod2':100000})
        
        tu.coords["nod2"] = ("nod2",np.arange(len(mld.nod2)))
        area.coords["nod2"] = ("nod2",np.arange(len(mld.nod2)))
        tu.coords["lon"] = mld.lon
        tu.coords["lat"] = mld.lat
        area.coords["lon"] = mld.lon
        area.coords["lat"] = mld.lat
        if mm == months[0]:
            mask_reg = maskS.isel(time=mm-1).reset_coords(drop=True).load()
            tu_reg = (tu*area.reset_coords(drop=True)
                       ).where((~mask_reg) & (depth < -2e2),
                               drop=True
                              )
            tu_reg = tu_reg.where((tu_reg.lat>so) & (tu_reg.lat<no)
                                  & (tu_reg.lon>we) & (tu_reg.lon<ea),
                                  drop=True
                                 )
            tu_reg.lon.plot()
            plt.show()
            tu_reg.lat.plot()
            plt.show()
            area_reg = area.where((~mask_reg) & (depth < -2e2),
                                 drop=True
                                )
            area_reg = area_reg.where((area_reg.lat>so) & (area_reg.lat<no)
                                    & (area_reg.lon>we) & (area_reg.lon<ea),
                                    drop=True
                                   ).reset_coords(drop=True)
            tu_reg = (tu_reg.sum("nod2",skipna=True)
                      / area_reg.sum("nod2",skipna=True)
                     )
            print(tu_reg)
        else:
            tmp_reg = (tu*area.reset_coords(drop=True)
                      ).where((~mask_reg) & (depth < -2e2),
                              drop=True
                             )
            tmp_reg = tmp_reg.where((tmp_reg.lat>so) & (tmp_reg.lat<no)
                                    & (tmp_reg.lon>we) & (tmp_reg.lon<ea),
                                    drop=True
                                   )
            # area_gs = area.where((~maskS.isel(time=mm-1).reset_coords(drop=True))
            #                      & (diag.zbar_n_bottom < -2e2),
            #                      drop=True
            #                     )
            # area_gs = area_gs.where((area.lat>31) & (area.lat<45)
            #                         & (area.lon>-80) & (area.lon<-60),
            #                         drop=True
            #                        ).reset_coords(drop=True)
            tmp_reg = (tmp_reg.sum("nod2",skipna=True)
                       / area_reg.sum("nod2",skipna=True)
                      )
            tu_reg = xr.concat([tu_reg, tmp_reg
                               ], "time")

            del tmp_reg
        mld.close()
        print("Month: ", mm)
    
    tu_reg.coords["time"] = ("time",JFMJAS)
    
    fig, ax = plt.subplots()
    ax.plot(tu_reg.isel(time=slice(None,3)).mean(["time"]), 
            tu_reg.nz1)
    # ax.set_xlim([ppdwp_gs.min(skipna=True).data,
    #              ppdwp_gs.max(skipna=True).data])
    ax.set_ylim([-1e3,0])
    plt.show()
    
    dsave = tu_reg.to_dataset(name="tu")
    dsave.reset_coords(drop=True).to_zarr(op.join(wdir,
                                                  "ke_adv/N%2d/GulfStrEx_%4d.zarr" 
                                                  % (NXX,yy)),
                                          mode="w")
    dsave.close()
    del tu_reg
    print("Year: ", yy)